# Logistic Regression — interpretable linear baseline

## Plan

1. **Preprocessing outputs and LR suitability.** The `linear` pipeline from notebook 02 is used unchanged. LR needs `log1p` and scaling because the counts are skewed up to 290, and it needs near-duplicate removal for stable coefficients. Both are done inside the pipeline, which is refit per CV fold.
2. **Feature inclusion and exclusion.** See the table below. It is based on the EDA statistics.
3. **Transformations.** Counts/bytes: `log1p` → `StandardScaler`. Rates: `StandardScaler`. Binary: passthrough. Categoricals: OHE with rare grouping (`min_frequency=50`).
4. **Pipeline.** `Pipeline[FeatureCleaner → CorrelationFilter(0.98) → ColumnTransformer → LogisticRegression]`.
5. **Search space (Optuna TPE, 30 trials, seed 42).**
   - `penalty` ∈ {L2 (`lbfgs`), L1 (`liblinear`)}
   - `C` ∈ [1e-3, 1e2], log-uniform
   - `class_weight` ∈ {None, balanced}

   Elastic net requires `saga`. On this data `saga` took 330 s per fit and still did not converge (max_iter=3000), so it is evaluated once as an ablation instead of inside the search.
6. **Strategy.**
   - Objective: stratified 5-fold CV macro-F1 on `train_sub` (80% of KDDTrain+).
   - Decision threshold chosen on `val` (20%).
   - Refit on all of KDDTrain+, then score KDDTest+ **once**.
7. **Experiments and ablations** (same CV folds):
   - without `log1p`
   - without the correlation filter
   - without rare grouping
   - `class_weight` flipped
   - L1 vs L2
   - elastic net

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src import experiment as ex, plots
from src.features import build_pipeline, build_preprocessor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

rng = ex.init()
SEED, CFG = ex.SEED, ex.CFG
sns.set_theme(style="whitegrid")
MODEL = "logistic_regression"
PREP = dict(corr_threshold=CFG["features"]["corr_threshold"], min_frequency=CFG["features"]["min_frequency"])

train, test, novel = ex.load_data()
idx_tr, idx_va = ex.get_splits(train)
X_all, y_all = ex.xy(train)
X_tr, y_tr = X_all.iloc[idx_tr], y_all[idx_tr]
run_id = ex.start_run(MODEL, {"family": "linear", "log_transform": True, **PREP})
print("MLflow run:", run_id, "| trials:", ex.n_trials())

## 1. Feature decisions (41 raw features)

| Group | Features | Decision | Evidence (EDA / notebook 02) |
|---|---|---|---|
| Basic (1–9) | `duration`, `src_bytes`, `dst_bytes`, `wrong_fragment`, `urgent` | Keep; `log1p` + scale | Skew 11–290; `src_bytes` has the top MI (0.56) |
| | `protocol_type`, `service`, `flag` | One-hot. `service`: 9 rare levels grouped. `flag`: 2 rare levels grouped. | Strong attack-rate differences; no unseen test levels |
| | `land` | Keep (binary) | 25 rows, 72% attack (LAND attack); L1 may zero it |
| Content (10–22) | `hot`, `num_failed_logins`, `num_compromised`, `num_file_creations`, `num_shells`, `num_access_files` | Keep; `log1p` + scale | Near-constant but carry R2L/U2R evidence |
| | `num_root` | **Drop** (correlation filter) | r = 0.999 with `num_compromised` |
| | `su_attempted` | Keep; 2 → 1 | 59 invalid values, all with `root_shell = 1` |
| | `logged_in`, `root_shell`, `is_host_login`, `is_guest_login` | Keep (binary) | `logged_in` MI 0.28; the rest are rare-attack flags |
| | `num_outbound_cmds` | **Drop** | Constant |
| Time traffic (23–31) | `count`, `srv_count` | Keep; `log1p` + scale | Capped at 511, skewed |
| | `srv_serror_rate`, `srv_rerror_rate` | **Drop** (correlation filter) | r = 0.993 / 0.989 with the host-level twin |
| | other 5 rates | Keep; scale | `diff_srv_rate`, `same_srv_rate` MI ≈ 0.36 |
| Host traffic (32–41) | `dst_host_count`, `dst_host_srv_count` | Keep; `log1p` + scale | Capped at 255 |
| | `dst_host_srv_serror_rate` | **Drop** (correlation filter) | r = 0.986 with `srv_serror_rate`, 0.981 with `serror_rate` |
| | other 7 rates | Keep; scale | |

**Not applied (justified):**
- No hand-binarisation of the near-constant counts: `log1p` already maps 0 → 0, so the binary information is preserved.
- No interaction terms or polynomial features: interpretability is the priority.
- No explicit feature selector: notebook 02 showed that MI and importance selectors do not beat "all features".

In [ ]:
def make_pipeline(p, max_iter=2000, **prep_over):
    solver = "lbfgs" if p["penalty"] == "l2" else "liblinear"
    est = LogisticRegression(C=p["C"], l1_ratio=0.0 if p["penalty"] == "l2" else 1.0, solver=solver,
                             class_weight=p["class_weight"], max_iter=max_iter, random_state=SEED)
    return build_pipeline(est, "linear", **{**PREP, **prep_over})

def suggest(trial):
    return {"penalty": trial.suggest_categorical("penalty", ["l2", "l1"]),
            "C": trial.suggest_float("C", 1e-3, 1e2, log=True),
            "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"])}

t0 = time.perf_counter()
study = ex.tune(MODEL, run_id, make_pipeline, suggest, X_tr, y_tr, cv_jobs=5)
tuning_seconds = time.perf_counter() - t0
best = study.best_params
print(f"best CV macro-F1 = {study.best_value:.4f} with {best}  ({tuning_seconds/60:.1f} min)")

In [ ]:
fig = plots.optuna_overview(study, "C", hue="penalty"); ex.log_figure(run_id, fig, "optuna", MODEL); plt.show()

## 2. Train / validation diagnostics, threshold and the single test evaluation

In [ ]:
res = ex.fit_and_evaluate(MODEL, run_id, make_pipeline, best, train, test, idx_tr, idx_va, novel,
                          extra={"tuning": {"best_cv_f1_macro": study.best_value, "n_trials": len(study.trials),
                                            "tuning_seconds": tuning_seconds}})
pd.DataFrame({k: res["metrics"][k] for k in ["train", "val", "test", "test_at_0.5"]}).T[
    ["threshold", "accuracy", "precision", "recall", "f1", "f1_macro", "roc_auc", "pr_auc", "fpr", "fnr"]].round(4)

In [ ]:
fig = plots.metric_bars(res["metrics"], "Logistic Regression: train vs validation vs test"); ex.log_figure(run_id, fig, "metric_bars", MODEL); plt.show()
fig = plots.confusion_and_curves(res["y_va"], res["p_va"], res["y_te"], res["p_te"], res["threshold"], "LR")
ex.log_figure(run_id, fig, "confusion_roc_pr_threshold", MODEL); plt.show()

## 3. Coefficient analysis

In [ ]:
final = res["final"]
names = final[:-1].get_feature_names_out()
coef = pd.Series(final[-1].coef_[0], index=[n.split("__", 1)[1] for n in names])
nz = int((coef != 0).sum())
top = coef.reindex(coef.abs().sort_values(ascending=False).index).head(20)
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.6, 1]})
axes[0].barh(top.index[::-1], top.values[::-1], color=np.where(top.values[::-1] > 0, "#C44E52", "#4C72B0"))
axes[0].axvline(0, c="k", lw=.8); axes[0].set_title(f"Top-20 coefficients (red → attack). Non-zero: {nz}/{len(coef)}")
axes[0].set_xlabel("coefficient (per SD for scaled features; per unit for dummies/flags)")
grp = pd.Series({n: n.split("__")[0] for n in names}).map({"log": "counts/bytes", "rate": "rates", "bin": "binary", "cat": "categorical"})
imp = pd.Series(np.abs(final[-1].coef_[0]), index=grp.values).groupby(level=0).sum()
axes[1].pie(imp.values, labels=imp.index, autopct="%1.0f%%", colors=sns.color_palette("muted"))
axes[1].set_title("Share of total |coefficient| by feature block")
fig.tight_layout(); ex.log_figure(run_id, fig, "coefficients", MODEL); plt.show()
odds = pd.DataFrame({"coef": top, "odds_ratio": np.exp(top)}).round(3)
odds.to_csv(PROJECT_ROOT / "results" / "tuning" / f"{MODEL}_top_coefficients.csv")
odds.head(10)

## 4. Error analysis (false negatives and false positives)

In [ ]:
fig = plots.error_analysis(res["preds"], test, "LR"); ex.log_figure(run_id, fig, "error_analysis", MODEL); plt.show()
fig = plots.missed_attack_types(res["preds"]); ex.log_figure(run_id, fig, "missed_attack_types", MODEL); plt.show()

## 5. Ablation studies (5-fold CV on `train_sub`, same folds, best hyperparameters)

In [ ]:
# Ablations use max_iter=300 for every variant (the tuned model converges in ~25 iterations).
# A first run with max_iter=2000 did not finish in 70 min: without log1p, liblinear L1 at high C
# fails to converge on the raw heavy-tailed counts. Non-convergence is recorded as a finding.
from sklearn.model_selection import cross_validate
from sklearn.exceptions import ConvergenceWarning
import warnings

def cv_score(pipe):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always", ConvergenceWarning)
        r = cross_validate(pipe, X_tr, y_tr, cv=ex.cv_splitter(), scoring="f1_macro", n_jobs=5, return_estimator=True)
    n_iter = int(max(np.max(e[-1].n_iter_) for e in r["estimator"]))
    return r["test_score"].mean(), r["test_score"].std(), n_iter, n_iter >= 300

AB = dict(max_iter=300)
variants = {
    "tuned (reference)": make_pipeline(best, **AB),
    "no log1p": make_pipeline(best, log_transform=False, **AB),
    "no correlation filter": make_pipeline(best, corr_threshold=None, **AB),
    "no rare-level grouping": make_pipeline(best, min_frequency=None, **AB),
    "class_weight flipped": make_pipeline({**best, "class_weight": None if best["class_weight"] else "balanced"}, **AB),
    "other penalty (L1↔L2)": make_pipeline({**best, "penalty": "l1" if best["penalty"] == "l2" else "l2"}, **AB),
}
abl = pd.DataFrame({k: cv_score(v) for k, v in variants.items()}, index=["cv_f1_macro", "std", "max_n_iter", "hit_max_iter"]).T
abl[["cv_f1_macro", "std"]] = abl[["cv_f1_macro", "std"]].astype(float)

# Elastic net (saga) once, on a 20k stratified subsample: fit on it, score on val
from sklearn.model_selection import train_test_split
sub, _ = train_test_split(np.arange(len(X_tr)), train_size=20000, stratify=y_tr, random_state=SEED)
en = build_pipeline(LogisticRegression(C=best["C"], l1_ratio=0.5, solver="saga", max_iter=1000, tol=1e-3,
                                       class_weight=best["class_weight"], random_state=SEED), "linear", **PREP)
X_va, y_va = X_all.iloc[idx_va], y_all[idx_va]
t0 = time.perf_counter(); en.fit(X_tr.iloc[sub], y_tr[sub]); en_s = time.perf_counter() - t0
ref = make_pipeline(best).fit(X_tr.iloc[sub], y_tr[sub])
from sklearn.metrics import f1_score
en_cmp = pd.Series({"elastic net (saga, l1_ratio=0.5)": f1_score(y_va, en.predict(X_va), average="macro"),
                    f"tuned {best['penalty'].upper()}": f1_score(y_va, ref.predict(X_va), average="macro")})

fig, axes = plt.subplots(1, 2, figsize=(16, 4), gridspec_kw={"width_ratios": [2, 1]})
a = abl.sort_values("cv_f1_macro")
axes[0].barh([f"{i} (no convergence)" if h else i for i, h in zip(a.index, a["hit_max_iter"])], a["cv_f1_macro"], xerr=a["std"],
             color=["#C44E52" if i == "tuned (reference)" else "#8C8C8C" for i in a.index])
axes[0].set_xlim(a["cv_f1_macro"].min() - .005, a["cv_f1_macro"].max() + .003); axes[0].set_title("Ablations: CV macro-F1")
axes[1].bar(en_cmp.index, en_cmp.values, color=["#8172B2", "#C44E52"]); axes[1].set_ylim(en_cmp.min() - .01, 1)
axes[1].set_title(f"Elastic net vs tuned (20k fit, val score; saga {en_s:.0f}s)"); axes[1].tick_params(axis="x", rotation=10)
fig.tight_layout(); ex.log_figure(run_id, fig, "ablations", MODEL); plt.show()

abl.to_csv(PROJECT_ROOT / "results" / "tuning" / f"{MODEL}_ablations.csv")
import mlflow
ex.setup_mlflow()
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics({f"ablation_{k.replace(' ', '_').replace('(', '').replace(')', '').replace('↔', '_')}": v
                        for k, v in abl["cv_f1_macro"].items()})
abl.round(4)

## 6. Summary

In [ ]:
m = res["metrics"]
summary = pd.DataFrame({
    "value": [str(best), f"{study.best_value:.4f}", f"{res['threshold']:.2f}",
              f"{m['train']['f1_macro']:.4f} / {m['val']['f1_macro']:.4f} / {m['test']['f1_macro']:.4f}",
              f"{m['test']['recall']:.4f}", f"{m['test']['fpr']:.4f}", f"{m['test']['roc_auc']:.4f}",
              f"{res['slices']['recall_novel']:.4f}", res["payload"]["complexity"].get("nonzero_coefficients"),
              f"{res['payload']['timing']['final_fit_seconds']:.1f}s", res["payload"]["artifacts"].get("onnx_status")]},
    index=["best params", "CV macro-F1", "threshold (val)", "macro-F1 train / val / test", "test attack recall",
           "test FPR", "test ROC-AUC", "novel-attack recall", "non-zero coefficients", "final fit time", "ONNX export"])
summary